# Spherinator & HiPSter: an interactive sky of simulated galaxies

This notebook walks through the complete
[Spherinator](https://github.com/HITS-AIN/Spherinator) /
[HiPSter](https://github.com/HITS-AIN/HiPSter) pipeline on the
[IllustrisTNG SKIRT SDSS](https://huggingface.co/datasets/HITS-AIN/IllustrisTNG_SKIRT_SDSS)
dataset of synthetic galaxy images:

1. **Load** the dataset with a Spherinator `DataModule`.
2. **Train** a variational autoencoder whose latent space is the surface of the
   unit sphere $S^2$ — a small convolutional encoder and an upsampling decoder.
3. **Export** the trained encoder and decoder to ONNX.
4. **Generate** a [HiPS](https://www.ivoa.net/documents/HiPS/) tiling with HiPSter:
   the decoder is evaluated at every HEALPix cell centre, so each tile shows what
   the model *imagines* at that point of the sphere.
5. **Explore** the result in [Aladin Lite](https://aladin.cds.unistra.fr/AladinLite/)
   through the `ipyaladin` widget, with the real galaxies overlaid as a catalogue.

The key idea is that a *spherical* latent space is directly a sky: once every
galaxy has a position on $S^2$, astronomical sky-viewers become general-purpose
tools for exploring a learned representation.


In [3]:
# The two project packages plus the notebook widget. Installed by `uv sync` from
# pyproject.toml; the fallback keeps the notebook runnable in a bare Jupyter
# container (see compose.yml).
try:
    import hipster
    import ipyaladin
    import spherinator
except ImportError:
    %pip -q install git+https://github.com/HITS-AIN/Spherinator git+https://github.com/HITS-AIN/HiPSter ipyaladin
    import hipster
    import ipyaladin
    import spherinator

print(f"spherinator {spherinator.__version__}")
print(f"hipster     {hipster.__version__}")
print(f"ipyaladin   {ipyaladin.__version__}")


spherinator 0.5.1
hipster     0.1.1
ipyaladin   0.8.0


In [ ]:
%matplotlib inline
import os

import lightning.pytorch as pl
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

# Every artefact this notebook produces lands under `output/`, which the HTTP
# server further down exposes to Aladin Lite at BASE_URL.
OUTPUT_PATH = "output"
ONNX_PATH = os.path.join(OUTPUT_PATH, "onnx")
PORT = 8083
BASE_URL = f"http://localhost:{PORT}"

# Fix all RNG seeds so reruns give the same model.
pl.seed_everything(42)

## 1. The dataset

`spherinator.data.DataModule` is a thin Lightning wrapper around a
[Hugging Face dataset](https://huggingface.co/docs/datasets). It streams the
requested `columns`, converts `uint8` image data to `float32` in $[0, 1]$, and
splits off validation and test sets.

The IllustrisTNG SKIRT SDSS dataset holds 42,998 galaxies from the
[IllustrisTNG](https://www.tng-project.org/) cosmological simulations, each
rendered with the [SKIRT](https://skirt.ugent.be/) radiative-transfer code into
SDSS photometric bands and stored as a 128 × 128 RGB image.

`return_dict=False` with a single column makes the dataloader yield bare image
tensors of shape `[batch, 3, 128, 128]`, which is exactly what the autoencoder
expects — no unpacking needed in the training loop.


In [5]:
datamodule = spherinator.data.DataModule(
    path="HITS-AIN/IllustrisTNG_SKIRT_SDSS",
    columns=["image"],
    return_dict=False,
    batch_size=64,
    shuffle=True,
    num_workers=4,
)


Let's look at the data before modelling it. `setup("fit")` downloads the dataset
(cached after the first call) and builds the train/validation split.


In [ ]:
datamodule.setup("fit")
images = next(iter(datamodule.train_dataloader()))
print(f"batch {tuple(images.shape)}, range [{images.min():.2f}, {images.max():.2f}]")

fig, axes = plt.subplots(5, 5, figsize=(10, 10))
for ax, image in zip(axes.flat, images):
    ax.imshow(image.permute(1, 2, 0))  # [3, 128, 128] -> [128, 128, 3] for imshow
    ax.axis("off")
fig.suptitle("IllustrisTNG SKIRT SDSS galaxies")
fig.tight_layout()
plt.show()


batch (64, 3, 128, 128), range [0.00, 1.00]


## 2. A variational autoencoder on the sphere

`spherinator.models.VariationalAutoencoder` differs from a textbook VAE in the
shape of its latent space. Instead of a Gaussian in $\mathbb{R}^d$ it uses a
[power spherical](https://arxiv.org/abs/2006.04437) distribution on the unit
sphere $S^{z_{dim}-1}$:

- The encoder maps an image to a feature vector of size `encoder_out_dim`.
- A `SphereHead` turns that vector into a **direction** `z_location`
  (L2-normalised, so it lies exactly on the sphere) and a **concentration**
  `z_scale` (how sharply peaked the distribution around that direction is).
- The KL term pulls the posterior towards the *uniform* distribution on the
  sphere, weighted by `beta`. Because the sphere is compact and has no
  distinguished origin, there is no "posterior collapse to zero" — the
  representation stays spread over the whole surface.
- The decoder maps a sampled direction back to an image.

With `z_dim=3` the latent space is the ordinary 2-sphere, which is precisely why
the result can be displayed as a sky.

### Why a small CNN?

The obvious encoder choice is a pretrained backbone such as
`HuggingFaceResNetEncoder("microsoft/resnet-18")`, but for 128 × 128 synthetic
galaxies that is far more capacity than the problem needs — ResNet-18 alone is
11.3 M parameters, and ImageNet features are not a natural prior for
low-surface-brightness astronomical images.

Instead we stack five plain convolution blocks built from
`ConsecutiveConv2DLayer`, each `Conv2d → BatchNorm2d → ReLU → MaxPool2d(2)`.
The spatial resolution halves and the channel count doubles at every step:

| stage | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| resolution | 64² | 32² | 16² | 8² | 4² |
| channels | 16 | 32 | 64 | 128 | 256 |

A final `Flatten → LazyLinear` projects the 256 × 4 × 4 feature map onto the
256-dimensional vector the `SphereHead` reads. `LazyConv2d`/`LazyLinear` infer
their input sizes on the first forward pass, so the layer list does not have to
repeat the shapes.

The decoder mirrors this: `Linear(3, 256)` lifts the latent direction, then
`UpsamplingDecoder2D` grows a 4 × 4 seed feature map to 128 × 128 by repeated
*bilinear upsampling followed by convolution* — which, unlike transposed
convolution, does not produce checkerboard artefacts. Its `base_channels=128`
(down from the default 512) keeps the decoder small too.

Together the two nets come to **≈ 2.1 M parameters instead of ≈ 15.0 M**, about
7× smaller. That trains in minutes on a small GPU and, just as importantly,
keeps the exported decoder cheap — HiPSter has to evaluate it once per HiPS
tile, thousands of times.


In [5]:
# Five Conv-BN-ReLU-MaxPool blocks: 128 -> 64 -> 32 -> 16 -> 8 -> 4 pixels.
encoder = spherinator.models.ConvolutionalEncoder2D(
    input_dim=[3, 128, 128],
    output_dim=256,
    cnn_layers=[
        spherinator.models.ConsecutiveConv2DLayer(
            out_channels=[channels],
            kernel_size=3,
            stride=1,
            padding=1,  # 'same' padding: only the pooling changes the resolution
            pooling=nn.MaxPool2d(2),
        )
        for channels in (16, 32, 64, 128, 256)
    ],
)

# Latent direction (3) -> feature vector (256) -> image (3, 128, 128).
decoder = spherinator.models.Sequential(
    modules=[
        nn.Linear(in_features=3, out_features=256),
        spherinator.models.UpsamplingDecoder2D(
            input_dim=256,
            output_dim=[3, 128, 128],
            base_channels=128,
            seed_size=4,  # 4 * 2**5 = 128 -> five upsampling blocks
        ),
    ]
)

model = spherinator.models.VariationalAutoencoder(
    encoder=encoder,
    decoder=decoder,
    encoder_out_dim=256,  # must match the encoder's output_dim
    z_dim=3,  # latent space is the 2-sphere
    beta=1e-6,  # weight of the KL term; small = reconstruction dominates
    reconstruction_loss=nn.L1Loss(),  # more robust than MSE on faint pixels
    max_scale=1e4,  # clamp the concentration for numerical stability
)


def count_parameters(module: nn.Module) -> float:
    return sum(p.numel() for p in module.parameters()) / 1e6


# LazyConv2d/LazyLinear only materialise their weights on the first forward pass,
# so run one before counting parameters.
model(torch.randn(2, 3, 128, 128))
print(f"encoder {count_parameters(encoder):.2f} M parameters")
print(f"decoder {count_parameters(decoder):.2f} M parameters")
print(f"total   {count_parameters(model):.2f} M parameters")


encoder 1.44 M parameters
decoder 0.65 M parameters
total   2.09 M parameters


## 3. Training

Nothing Spherinator-specific here — the model is a `LightningModule`, so the
standard `Trainer` drives it. `precision="16-mixed"` roughly halves the memory
footprint and speeds up the convolutions on any recent NVIDIA GPU.

Metrics are written to `lightning_logs/`; `train_loss_recon` is the part you
want to watch, since with `beta=1e-6` the KL term is numerically tiny.


In [ ]:
trainer = pl.Trainer(
    max_epochs=10,
    accelerator="auto",
    precision="16-mixed",
    log_every_n_steps=50,
    limit_train_batches=1,  # fast smoke test
    limit_val_batches=1,  # fast smoke test
)
trainer.fit(model, datamodule=datamodule)


### Reconstructions

The honest check of an autoencoder: put images in, compare what comes out.
`model.reconstruct()` takes the *deterministic* path — it decodes `z_location`
directly rather than a sample from the posterior — which is also what the
exported ONNX graphs will do.

A 3-dimensional latent space is a brutal bottleneck: two angles per galaxy. So
do not expect pixel-perfect copies. What the reconstructions *should* capture is
morphology — size, elongation, orientation, colour, bulge-versus-disc — because
those are the dimensions the model has to spend its two angles on.


In [7]:
model.eval()
images = next(iter(datamodule.val_dataloader()))[:8]
with torch.no_grad():
    reconstruction = model.reconstruct(images.to(model.device)).cpu()

fig, axes = plt.subplots(2, 8, figsize=(16, 4.4))
for column, (original, recon) in enumerate(zip(images, reconstruction)):
    axes[0, column].imshow(original.permute(1, 2, 0))
    axes[1, column].imshow(recon.clamp(0, 1).permute(1, 2, 0))
    for row in (0, 1):
        axes[row, column].axis("off")
axes[0, 0].set_title("original", loc="left")
axes[1, 0].set_title("reconstruction", loc="left")
fig.tight_layout()
plt.show()


/tmp/ipykernel_198515/256197493.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Where do the galaxies land?

Before generating a whole sky, it is worth confirming that the encoder actually
*uses* the sphere. If training had collapsed, every galaxy would sit in one
small patch and the HiPS tiling would be uniform mush.

The plot below encodes a few thousand galaxies and shows their latent directions
in a Mollweide projection — the same projection an all-sky map uses.


In [8]:
latent = []
with torch.no_grad():
    for batch, _ in zip(datamodule.val_dataloader(), range(40)):
        z_location, _ = model.encode(batch.to(model.device))
        latent.append(z_location.cpu().numpy())
latent = np.concatenate(latent)

# Cartesian direction on the unit sphere -> longitude/latitude in radians.
longitude = np.arctan2(latent[:, 1], latent[:, 0])
latitude = np.arcsin(np.clip(latent[:, 2], -1.0, 1.0))

fig = plt.figure(figsize=(10, 5))
ax = fig.add_subplot(111, projection="mollweide")
ax.scatter(longitude, latitude, s=2, alpha=0.3)
ax.grid(True)
ax.set_title(f"Latent directions of {len(latent)} galaxies")
plt.show()


/tmp/ipykernel_198515/3906040309.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Export to ONNX

HiPSter does not import the PyTorch model: it loads
[ONNX](https://onnx.ai/) graphs through `onnxruntime`. That decoupling is
deliberate — tile generation needs no autograd, no Lightning, and no GPU, so it
can run wherever the tiles are being published.

Two graphs are needed, and each gets a thin `nn.Module` wrapper because
`VariationalAutoencoder.forward` returns the whole training tuple
(distributions, samples, reconstruction) rather than a single tensor:

| graph | input | output | used for |
|---|---|---|---|
| `encoder.onnx` | image `x` `[N, 3, 128, 128]` | direction `[N, 3]` | placing real galaxies on the sky |
| `decoder.onnx` | direction `z` `[N, 3]` | image `[N, 3, 128, 128]` | painting the HiPS tiles |

Two details matter for HiPSter:

- **The input name is the `forward` argument name.** HiPSter's `Inference` class
  passes the input under a configurable `input_name` — hence `forward(self, x)`
  for the encoder and `forward(self, z)` for the decoder.
- **The batch axis must be dynamic.** `dynamic_shapes={"z": {0: "batch"}}` marks
  it as such; without it the graph is frozen at the batch size of the example
  input, and HiPSter calls the decoder with `hierarchy²` rows at a time.

`spherinator.models.export_onnx()` does all of this from a checkpoint plus a
model YAML file, which is the right entry point in a scripted workflow. Here the
trained model is already in memory, so we call `torch.onnx.export` directly.


In [9]:
class EncoderONNX(nn.Module):
    """Image -> latent direction, taking the deterministic mean of the posterior."""

    def __init__(self, model: nn.Module) -> None:
        super().__init__()
        self.model = model

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z_location, _z_scale = self.model.sphere_head(self.model.encoder(x))
        return z_location


class DecoderONNX(nn.Module):
    """Latent direction -> image."""

    def __init__(self, model: nn.Module) -> None:
        super().__init__()
        self.model = model

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        return self.model.decoder(z)


os.makedirs(ONNX_PATH, exist_ok=True)
model = model.to("cpu").eval()

with torch.no_grad():
    exported = torch.onnx.export(
        EncoderONNX(model),
        (torch.randn(1, 3, 128, 128),),
        dynamic_shapes={"x": {0: "batch"}},
        dynamo=True,
        opset_version=19,
    )
    exported.optimize()
    exported.save(os.path.join(ONNX_PATH, "encoder.onnx"))

    exported = torch.onnx.export(
        DecoderONNX(model),
        (torch.randn(1, 3),),
        dynamic_shapes={"z": {0: "batch"}},
        dynamo=True,
        opset_version=19,
    )
    exported.optimize()
    exported.save(os.path.join(ONNX_PATH, "decoder.onnx"))


/tmp/ipykernel_198515/3418583807.py:28: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  exported = torch.onnx.export(


[torch.onnx] Obtain model graph for `EncoderONNX([...]` with `torch.export.export(..., strict=False)`...


[torch.onnx] Obtain model graph for `EncoderONNX([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/home/doserbd/.local/share/uv/python/cpython-3.13.2-linux-x86_64-gnu/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
/tmp/ipykernel_198515/3418583807.py:38: UserWarning: Exporting a model while it is in training mode. Please ensure that this is intended, as it may lead to different behavior during inference. Calling model.eval() before export is recommended.
  exported = torch.onnx.export(


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


[torch.onnx] Obtain model graph for `DecoderONNX([...]` with `torch.export.export(..., strict=False)`...


[torch.onnx] Obtain model graph for `DecoderONNX([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅


/home/doserbd/.local/share/uv/python/cpython-3.13.2-linux-x86_64-gnu/lib/python3.13/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


Verify the exported graphs: the declared signatures should show a symbolic
`batch` dimension, and running the ONNX encoder must reproduce the PyTorch
output to floating-point tolerance.


In [10]:
import onnxruntime as ort

for name in ("encoder.onnx", "decoder.onnx"):
    session = ort.InferenceSession(os.path.join(ONNX_PATH, name))
    inputs = [(i.name, i.shape) for i in session.get_inputs()]
    outputs = [o.shape for o in session.get_outputs()]
    print(f"{name}: {inputs} -> {outputs}")

# Same input through both runtimes -> same latent direction?
sample = next(iter(datamodule.val_dataloader()))[:16]
with torch.no_grad():
    expected = EncoderONNX(model)(sample).numpy()
session = ort.InferenceSession(os.path.join(ONNX_PATH, "encoder.onnx"))
actual = session.run(None, {"x": sample.numpy()})[0]
print(f"max |torch - onnx| = {np.abs(expected - actual).max():.2e}")


encoder.onnx: [('x', ['batch', 3, 128, 128])] -> [['batch', 3]]
decoder.onnx: [('z', ['batch', 3])] -> [['batch', 3, 128, 128]]


max |torch - onnx| = 3.28e-07


## 5. Generate the HiPS tiling with HiPSter

[HiPS](https://www.ivoa.net/documents/HiPS/) (Hierarchical Progressive Survey)
is the IVOA standard behind interactive all-sky viewers. The sphere is split
into 12 base [HEALPix](https://healpix.sourceforge.io/) cells, and each order
subdivides every cell into four: order $k$ has $12 \cdot 4^k$ tiles. A viewer
downloads only the tiles for the current field of view, so panning and zooming
stay cheap no matter how deep the tiling goes.

`hipster.HiPSGenerator` fills that structure straight from the model. For every
HEALPix cell it takes the cell centre as a unit vector — which *is* a point in
the latent space — feeds it to the decoder, and stores the generated image as
that tile. The resulting sky is a continuous atlas of galaxy morphology: nearby
tiles show similar galaxies because the latent space is smooth.

The pieces:

- `Inference` wraps an ONNX graph; `input_name="z"` matches `DecoderONNX.forward`.
- `ImagePlotter` converts a decoder output `[3, H, W]` in $[0, 1]$ into a
  `uint8` RGB array `[H, W, 3]`.
- `hierarchy=2` packs a 2 × 2 block of decoded images into each stored tile, so
  one HTTP request delivers four latent samples: 128 px images become 256 px
  tiles.
- `max_order=3` gives $12 \cdot (1 + 4 + 16 + 64) = 1020$ tiles, a couple of
  minutes of decoding. Raise it for a sharper sky — each extra order costs 4×.
- `distortion_correction=True` resamples each image so that it lines up with the
  actual curved shape of its HEALPix cell instead of being naively squeezed in.

`HTMLGenerator` writes a standalone `index.html` that loads the tiling in Aladin
Lite. Every task `register`s itself with it, so the page knows which layers
exist — we add the galaxy catalogue as a second layer further down. Running the
same thing from the shell is `hipster --config <config.yaml>`; the YAML mirrors
these arguments one-to-one (see
[`examples/`](https://github.com/HITS-AIN/HiPSter/tree/main/examples)).


In [11]:
# `url` is where the browser will reach these files - see the HTTP server below.
html_generator = hipster.HTMLGenerator(
    root_path=OUTPUT_PATH,
    url=BASE_URL,
    title="Spherinator model of IllustrisTNG SKIRT SDSS",
    aladin_lite_version="3.6.5",
)

hips_generator = hipster.HiPSGenerator(
    decoder=hipster.Inference(
        model_path=os.path.join(ONNX_PATH, "decoder.onnx"),
        input_name="z",
    ),
    image_maker=hipster.ImagePlotter(),
    max_order=3,
    hierarchy=2,
    hips_id="IllustrisTNG_SKIRT_SDSS",
    hips_name="IllustrisTNG SKIRT SDSS model",
    hips_path="model",
    root_path=OUTPUT_PATH,
    distortion_correction=True,
)

hips_generator.register(html_generator)
hips_generator.execute()
html_generator.generate()

print(open(os.path.join(OUTPUT_PATH, "model", "properties")).read())


Executing task: HiPSGenerator


Generating HTML page...

creator_did          = ivo://HITS/hipster
obs_title            = IllustrisTNG SKIRT SDSS model
dataproduct_type     = image
dataproduct_subtype  = color
hips_version         = 1.4
hips_creation_date   = 2026-09-03T09:01:42.110409+00:00
hips_status          = public master clonable
hips_tile_format     = jpeg
hips_order           = 3
hips_order_min       = 0
hips_tile_width      = 256
hips_frame           = equatorial



`HiPSGenerator` also stitches an `Allsky.jpg` — a single image holding every
tile of one order. Aladin Lite grabs it first to paint the whole sphere at once
before requesting individual tiles, and it doubles as a compact overview of what
the decoder learned. Each little square is one point of the latent sphere.


In [12]:
allsky = plt.imread(os.path.join(OUTPUT_PATH, "model", "Norder3", "Allsky.jpg"))
plt.figure(figsize=(9, 9))
plt.imshow(allsky)
plt.axis("off")
plt.title("All-sky view of the decoded latent sphere (order 3)")
plt.show()


/tmp/ipykernel_198515/2059114969.py:6: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. A catalogue of the real galaxies

The tiling shows what the *model* generates. To see where the *data* sits, we
run the ONNX encoder over the whole dataset and convert each latent direction to
sky coordinates.

The conversion is `healpy.vec2ang`, which turns a unit vector into HEALPix
spherical coordinates $(\theta, \phi)$: colatitude $\theta$ measured from the
north pole, longitude $\phi$. Right ascension is $\phi$, and declination is
$90° - \theta$.

This second `DataModule` also carries the provenance columns
(`simulation`, `snapshot`, `subhalo_id`), so clicking a source in Aladin
identifies the actual subhalo. `validation_size=0.0` and `shuffle=False` keep
the full dataset in its original order. HiPSter ships
`hipster.VOTableGenerator` for this step when reading local parquet files with a
flat image column; here we build the table directly from the Hugging Face
dataset.


In [ ]:
catalog_datamodule = spherinator.data.DataModule(
    path="HITS-AIN/IllustrisTNG_SKIRT_SDSS",
    columns=["image", "simulation", "snapshot", "subhalo_id"],
    return_dict=True,  # we need the metadata alongside the images
    validation_size=0.0,  # no split: encode every galaxy
    test_size=0.0,
    batch_size=256,
    shuffle=False,
    num_workers=4,
)
catalog_datamodule.setup("fit")

encoder_inference = hipster.Inference(
    model_path=os.path.join(ONNX_PATH, "encoder.onnx"),
    input_name="x",
)

latent, metadata = [], {"simulation": [], "snapshot": [], "subhalo_id": []}
for batch in catalog_datamodule.train_dataloader():
    latent.append(encoder_inference(batch["image"].numpy()))
    for column in metadata:
        values = batch[column]
        metadata[column].extend(
            values.tolist() if torch.is_tensor(values) else list(values)
        )
latent = np.concatenate(latent)
print(f"encoded {len(latent)} galaxies")


encoded 42998 galaxies


In [ ]:
import healpy
from astropy.table import Table

theta, phi = healpy.vec2ang(latent)  # colatitude and longitude, in radians

catalog = Table(
    {
        "ra": np.degrees(phi),
        "dec": 90.0 - np.degrees(theta),
        "simulation": metadata["simulation"],
        "snapshot": metadata["snapshot"],
        "subhalo_id": metadata["subhalo_id"],
    }
)
catalog["ra"].unit = "deg"
catalog["dec"].unit = "deg"
# UCDs let any VO client recognise these two columns as the sky position.
catalog["ra"].meta["ucd"] = "pos.eq.ra;meta.main"
catalog["dec"].meta["ucd"] = "pos.eq.dec;meta.main"

catalog.write(
    os.path.join(OUTPUT_PATH, "catalog.vot"), format="votable", overwrite=True
)

# Register the catalogue as a layer too, and re-render the standalone page so it
# offers both the model tiling and the real galaxies.
html_generator.add_votable(
    html_generator.VOTable(
        url=f"{BASE_URL}/catalog.vot",
        name="IllustrisTNG galaxies",
        color="#ff3b30",
        shape="circle",
        size=8,
    )
)
html_generator.generate()

catalog[:5]


Generating HTML page...


ra,dec,simulation,snapshot,subhalo_id
deg,deg,,,
float32,float32,str9,int64,int64
307.1885,-44.527756,TNG100,99,613338
338.07568,-36.216576,Illustris,135,379786
356.2814,-20.21698,Illustris,131,125162
337.71725,-22.584404,Illustris,135,492067
254.47693,17.820534,Illustris,131,487120


## 7. Serve the tiles

Aladin Lite runs in the browser, so it fetches tiles over HTTP rather than from
the notebook's filesystem. A `SimpleHTTPRequestHandler` on `output/` is enough,
with one addition: the page lives on the Jupyter origin while the tiles come
from port 8083, so the responses need
`Access-Control-Allow-Origin` — WebGL refuses to texture cross-origin images
without it, and the `properties` file is read with `fetch`.

The server runs on a daemon thread, so it goes away with the kernel.
`allow_reuse_address` lets the cell be re-run without waiting for the socket's
`TIME_WAIT` to expire.


In [15]:
import functools
import http.server
import socketserver
import threading


class CORSRequestHandler(http.server.SimpleHTTPRequestHandler):
    """Static file handler that allows cross-origin reads and stays quiet."""

    def end_headers(self) -> None:
        self.send_header("Access-Control-Allow-Origin", "*")
        super().end_headers()

    def log_message(self, *args) -> None:
        pass


class ReusableThreadingServer(socketserver.ThreadingTCPServer):
    allow_reuse_address = True
    daemon_threads = True


# Shut down a server left behind by an earlier run of this cell.
if (previous := globals().get("httpd")) is not None:
    previous.shutdown()
    previous.server_close()

httpd = ReusableThreadingServer(
    ("", PORT),
    functools.partial(CORSRequestHandler, directory=os.path.abspath(OUTPUT_PATH)),
)
threading.Thread(target=httpd.serve_forever, daemon=True).start()
print(f"serving {os.path.abspath(OUTPUT_PATH)} on {BASE_URL}")


serving /home/doserbd/git/Tutorial-Spherinator/jupyter/output on http://localhost:8083


## 8. Explore it with ipyaladin

`ipyaladin` embeds Aladin Lite as a Jupyter widget. Two calls do the work:

- `survey` — the base image layer. Point it at the HiPS directory and Aladin
  reads `properties` to discover the tile format, size and maximum order.
- `add_catalog_from_URL` — loads the VOTable written above from the same server.
  Reading it by URL keeps the widget's message channel free; the in-memory
  equivalent is `aladin.add_table(catalog, ...)`, which is more convenient for
  small tables but ships every row through the kernel-browser connection.
  Option names are converted to Aladin Lite's camelCase, so `source_size`
  becomes `sourceSize`; `on_click="showTable"` makes a click show the full
  catalogue row in a popup and also fills `aladin.clicked_object`.

`fov=180` starts zoomed out to the whole sphere. Then: **scroll** to zoom,
**drag** to pan, and click a marker to see which subhalo it is. The background
you are flying over is entirely synthetic — every pixel was produced by the
decoder from the latent direction at that point on the sky — while the markers
are the 42,998 real simulated galaxies at the positions the encoder assigned
them.

If the widget stays blank: the widget pulls Aladin Lite itself from a CDN, so it
needs internet access, and the tiles must be reachable from the browser — check
that the server cell above is still running, and if Jupyter runs in the
container from `compose.yml`, that port 8083 is published.


In [16]:
from ipyaladin import Aladin

aladin = Aladin(
    survey=f"{BASE_URL}/model",  # the HiPS tiling generated above
    target="0 +0",
    fov=180,  # degrees: start with the whole sphere in view
    height=600,
)
aladin


In [17]:
# Overlay the real galaxies. Aladin picks up 'ra'/'dec' from the UCDs, but naming
# the fields explicitly is robust against differently named columns.
_ = aladin.add_catalog_from_URL(
    f"{BASE_URL}/catalog.vot",
    {
        "name": "IllustrisTNG galaxies",
        "color": "#ff3b30",
        "shape": "circle",
        "source_size": 8,
        "on_click": "showTable",
        "ra_field": "ra",
        "dec_field": "dec",
    },
)


In [18]:
# Click a marker in the widget above, then re-run this cell.
aladin.clicked_object


{}

## Where to go next

`output/index.html` is a standalone page generated by `HTMLGenerator` — open
<http://localhost:8083/index.html> for the same view outside the notebook, ready
to be published to any static web host.

Some directions worth trying:

- **Sharper sky.** Raise `max_order` in `HiPSGenerator`; each order quadruples
  the tile count and the decoding time.
- **Longer training.** Ten epochs is a demo. Watch `train_loss_recon` in
  `lightning_logs/` and keep going while it falls.
- **A different bottleneck.** `z_dim=3` is what makes the latent space a *sky*,
  but Spherinator supports any $S^{n-1}$; higher `z_dim` reconstructs better at
  the cost of direct visualisability.
- **Stronger regularisation.** Increase `beta` to spread the galaxies more
  uniformly over the sphere, or use `spherinator.callbacks.KLAnnealing` to ramp
  it up during training.
- **Other encoders.** `HuggingFaceResNetEncoder` and `HuggingFaceViTEncoder`
  swap in as drop-in replacements if you want to trade size for accuracy.
- **Your own data.** Point `DataModule` at any Hugging Face dataset or local
  parquet files; nothing below the data loading is dataset-specific.

Further reading: Polsterer, Doser, Fehlner & Trujillo-Gomez,
[*Spherinator and HiPSter: Representation Learning for Unbiased Knowledge
Discovery from Simulations*](https://arxiv.org/abs/2406.03810) (2024), and the
[Spherinator documentation](https://spherinator.readthedocs.io).
